# MoE SFT — Mixture of Experts Fine-tuning

This notebook is **proven to work** by a smoke test (`smoke/07_moe_smoke.py`).

## Honest Warning: 16GB VRAM Limit

Production MoE models **cannot be trained** on 16GB — Unsloth doesn't support bnb-4bit on MoE expert parameters:

| Model | Total/Active | 16GB? | Reason |
|---|---|---|---|
| Qwen3-30B-A3B | 30B/3.3B | NO | no bnb-4bit -> 17.5GB minimum (bf16) |
| Qwen3.5-35B-A3B | 35B/3B | NO | 74GB |
| Gemma-4-26B-A4B-it | 26B/4B | NO | no bnb-4bit |
| Qwen3-16B-A3B | 16B/3B | NO | only GGUF on the Hub |
| **`imdatta0/tiny_qwen3_moe_2.8B_0.7B`** | **2.8B/0.7B** | YES | T4-compatible teaching model |

## This Notebook
- **Teaches the pipeline** — real MoE patterns
- Uses TinyQwen3 MoE (the same T4-compatible model Unsloth's own `TinyQwen3_MoE.ipynb` uses)
- For production with Qwen3-30B-A3B you need an **A100/H100** — the code is identical, only the model name changes

## Smoke Test Results (RTX 4070 Ti SUPER 16GB)
- Peak VRAM: **9.64 GB** (comfortably under 16GB)
- Pipeline ran on TRL 0.24 (the official notebooks pin `trl==0.22.2` — our stack is compatible)
- 3 steps / Train loss: 12.1 (model weights are teaching-grade, this isn't real training)

## Contents
1. **What is MoE?** — Routing + Sparse Experts
2. **Setup + `UNSLOTH_MOE_DISABLE_AUTOTUNE`**
3. **Model — TinyQwen3 MoE (2.8B/0.7B)**
4. **MoE-Specific LoRA** — adding `gate_up_proj` to target_modules
5. **Dataset — OpenMathReasoning-mini**
6. **SFTTrainer — standard**
7. **Training**
8. **Moving to Production — with A100**
9. **Common Pitfalls**

## 1. What is MoE?

**Mixture of Experts** — every layer of the model has **N experts**, and a **router** picks the best K of them for each token.

### Structure
```
[Input Token] -> Router -> [Expert 1, 2, ..., N]
                  v
        Best K experts selected (typically top-2)
                  v
        Only those K experts do the computation
                  v
             [Output Token]
```

### Advantages
- **Many parameters but little compute** — Qwen3-30B-A3B has 30B parameters but uses only **3.3B** per token
- **Faster inference** — compute like a small dense model, knowledge like a large one

### Disadvantages
- **VRAM = total params**, compute = active params -> no memory savings
- **Routing overhead** — extra cost on small batches
- **Expert imbalance** — some experts get used rarely; balancing is hard

### Unsloth's MoE Optimization (Faster MoE 2026)
- **Split LoRA reordering** — `Y = X @ loraA; Z = Y @ loraB` (no full delta materialization)
- **3 backends:** `grouped_mm` (default), `unsloth_triton`, `native_torch`
- **2x speed + 35% less VRAM** vs vanilla transformers MoE
- **12x speed** vs older transformers v4

## 2. Setup

**Critical env var:** `UNSLOTH_MOE_DISABLE_AUTOTUNE=1` — autotune eats memory and must be disabled on small GPUs.

In [ ]:
import os
os.environ['UNSLOTH_MOE_DISABLE_AUTOTUNE'] = '1'    # MUST be set BEFORE the unsloth import

import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 3. Model — TinyQwen3 MoE

**Two CRITICAL settings:**
- `load_in_4bit=False` — bnb-4bit is **not supported** on MoE expert parameters (HARD RULE)
- `fast_inference=False` — vLLM **not yet** available for MoE

For 16GB this small model is the only option. For production with Qwen3-30B-A3B just change the model name (requires A100+).

In [ ]:
MODEL = 'imdatta0/tiny_qwen3_moe_2.8B_0.7B'         # T4-compatible
max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL,
    max_seq_length = max_seq_length,
    load_in_4bit = False,                            # MoE: NO 4-bit
    fast_inference = False,                          # MoE: NO vLLM
)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params')

## 4. MoE-Specific LoRA

**Two differences** from standard SFT:

1. **Add `gate_up_proj` to the `target_modules` list** — these are the MoE expert MLPs
2. **`lora_alpha = r * 2`** — official notebook recipe (r=32 -> alpha=64)

**The router is NOT trained by default** — Unsloth keeps router fine-tuning disabled (`use_router=False` implicit). Fine-tuning the router risks expert imbalance and is generally not recommended.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
        'gate_up_proj',                               # MoE expert layers — CRITICAL
    ],
    lora_alpha = lora_rank * 2,                       # 2x — official recipe
    use_gradient_checkpointing = True,
    random_state = 3407,
    bias = 'none',
)

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template='qwen-3')

## 5. Dataset — OpenMathReasoning-mini

All official MoE notebooks use the same dataset. Math reasoning + DeepSeek-R1 traces (`<think>...`).

In [ ]:
dataset = load_dataset('unsloth/OpenMathReasoning-mini', split='cot')
print(f'Total rows: {len(dataset)}')

def to_messages(example):
    return {'conversations': [
        {'role': 'user', 'content': example['problem']},
        {'role': 'assistant', 'content': example['generated_solution']},
    ]}
dataset = dataset.map(to_messages, remove_columns=dataset.column_names)

def fmt(examples):
    return {'text': [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples['conversations']
    ]}
dataset = dataset.map(fmt, batched=True)
print(f'\nFirst 200 chars:')
print(dataset[0]['text'][:200])

## 6. SFTTrainer — Standard

There is **no MoE-specific** SFTConfig setting — just normal TRL config.

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        max_steps = 50,                              # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = 3407,
        report_to = 'none',
    ),
)

## 7. Training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name} | start mem = {start_mem} / {max_mem} GB')

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n=== RESULTS ===")
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.2f} sec")
print(f"Train loss:    {trainer_stats.metrics['train_loss']:.4f}")
print(f"Peak VRAM:     {used} GB")

## 8. Moving to Production — with A100

**The code is identical**, only the model name changes:

```python
# Production (A100 80GB)
MODEL = 'unsloth/Qwen3-30B-A3B-Instruct-2507'
max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL,
    max_seq_length = max_seq_length,
    load_in_4bit = False,                            # SAME — no MoE 4-bit
    fast_inference = False,
)
# get_peft_model + SFTTrainer all the same
```

### Production Hardware Estimates (Official Unsloth)
| Model | bf16 LoRA VRAM |
|---|---|
| Qwen3-30B-A3B | 17.5 GB (best case) |
| Qwen3-30B-A3B-Instruct-2507 | ~63 GB |
| Qwen3.5-35B-A3B | 74 GB |
| Qwen3-235B-A22B | 256+ GB |
| Qwen3.5-122B-A10B | 256 GB |

**A100 40GB:** Qwen3-30B-A3B base fits.
**A100 80GB:** Qwen3-30B-A3B-Instruct-2507 comfortably.
**H100/H200:** Qwen3.5-35B-A3B.

## 9. Common Pitfalls

| # | Mistake | Result | Fix |
|---|---|---|---|
| 1 | `load_in_4bit=True` on MoE | NotImplementedError | `False` is mandatory |
| 2 | `fast_inference=True` | Crash | `False` is mandatory |
| 3 | `gate_up_proj` missing from `target_modules` | Expert MLPs aren't trained | Must be in the list |
| 4 | `UNSLOTH_MOE_DISABLE_AUTOTUNE=0` (default) | Autotune causes memory peaks | Set `=1` |
| 5 | Trying to use the `Qwen3.5-7B-A1B` model | DOES NOT exist on the Hub | Smallest = Qwen3-30B-A3B |
| 6 | `Qwen3-16B-A3B` safetensors | GGUF only, can't be fine-tuned | Use 30B-A3B with A100+ |
| 7 | Fine-tuning the router | Expert imbalance, model collapse | Leave it off by default |
| 8 | Official notebooks pin `trl==0.22.2` | Use the older API | Our stack ran on `trl 0.24` (proven by smoke) |
| 9 | `lora_alpha = r` (standard SFT) | Suboptimal speed | For MoE use `r*2` |
| 10 | Multi-GPU MoE training | Not yet supported | Single-GPU only |

## Summary

1. **MoE = sparse experts + a router** — large model, little compute, but **VRAM = total params**
2. **No bnb-4bit for MoE** -> limited to 16GB
3. **TinyQwen3 MoE = T4-compatible teaching model** — real pattern, small model
4. **Add `gate_up_proj` to target_modules** — the MoE expert MLPs
5. **`lora_alpha = r * 2`** — official MoE recipe
6. **`UNSLOTH_MOE_DISABLE_AUTOTUNE=1`** is mandatory on consumer GPUs
7. **Production = A100+** — same code, model name changes
8. **Router off by default** — safer to leave it un-tuned

**Tested:** `smoke/07_moe_smoke.py` — pipeline ran on TRL 0.24, 9.64 GB VRAM (RTX 4070 Ti SUPER 16GB).

## Next step: Production MoE Training
1. Rent an A100/H100 (vast.ai, Lambda, RunPod, etc.)
2. Change `MODEL = 'unsloth/Qwen3-30B-A3B-Instruct-2507'`
3. Switch `max_steps = 50` to `num_train_epochs = 1`
4. Same code, real model